# Evaluation

Evaluating solutions of ill-defined problems using FP2MP-Eval approach

In [1]:
PROBLEMS_FILE = './data/problems.json'
SOLUTIONS_FILE = './data/solutions.json'
EVALUATIONS_FILE = './data/evaluations.json'
MODEL='openai/gpt-4.1'
N_JUDGES = 5
MAX_WORKERS = 5

In [2]:
from src import read_json, write_json, generate_id

problems = read_json(PROBLEMS_FILE)
solutions = read_json(SOLUTIONS_FILE)
evaluations = read_json(EVALUATIONS_FILE) or {}

Creating tasks for solutions not evaluated in `evaluations.json`

In [3]:
tasks = []

for solution_id, solution_data in solutions.items():
    evaluation_id = generate_id(solution_id, MODEL)
    if evaluation_id not in evaluations:
        problem_id = solution_data['problem_id']
        problem_content = problems[problem_id]['content']
        solution_content = solution_data['content']
        tasks.append({
            'evaluation_id': evaluation_id,
            'solution_id': solution_id,
            'problem_content': problem_content,
            'solution_content': solution_content
        })

len(tasks)

60

In [4]:
from tqdm import tqdm
from fp2mp_eval.core import FP2MPEval

fp2mp_eval = FP2MPEval(MODEL, n_judges=N_JUDGES)

for task in tqdm(tasks):
    evaluation_id = task['evaluation_id']
    solution_id = task['solution_id']
    problem_content = task['problem_content']
    solution_content = task['solution_content']

    case = (problem_content, solution_content)
    evals = fp2mp_eval.evaluate_case(case, MAX_WORKERS)
    
    evaluations[evaluation_id] = {
        'solution_id': solution_id,
        'model': MODEL,
        'content': [e.model_dump() for e in evals]
    }

100%|██████████| 60/60 [14:08<00:00, 14.13s/it]


In [5]:
write_json(evaluations, EVALUATIONS_FILE)